# Node2Vec Link Prediction

In [15]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import scipy.sparse as sp
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
import pickle
        

In [16]:
# rebuild knowledge graph
triples = r"C:\Users\burit\Documents\GitHub\CADA\model\gene_hpo_withoutpatient\all.triples"
edges = []
with open(triples, 'r') as f:
    content = f.read().splitlines()
    content = [x.split('\t') for x in content]
    for triple in content:
        triple.pop(1)
        triple = tuple(triple)
        edges.append(triple)

G = nx.Graph()
G.add_edges_from(edges)
Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
g = G.subgraph(Gcc[0])


In [17]:
from CADA.gae.preprocessing import mask_test_edges
np.random.seed(0) # make sure train-test split is consistent between notebooks
adj_sparse = nx.to_scipy_sparse_matrix(g)

# Perform train-validation-test split
adj_train, train_edges, train_edges_false, val_edges, val_edges_false, \
    test_edges, test_edges_false = mask_test_edges(adj_sparse, test_frac=.3, val_frac=.1, prevent_disconnect=False, verbose=True)
g_train = nx.from_scipy_sparse_matrix(adj_train) # new graph object with only non-hidden edges

preprocessing...
generating test/val sets...
creating false test edges...
creating false val edges...
creating false train edges...
final checks for disjointness...
creating adj_train...
Done with train-test split!



In [18]:
# Inspect train/test split
print("Total nodes:", adj_sparse.shape[0])
print("Total edges:", int(adj_sparse.nnz/2)) # adj is symmetric, so nnz (num non-zero) = 2*num_edges
print("Training edges (positive):", len(train_edges))
print("Training edges (negative):", len(train_edges_false))
print("Validation edges (positive):", len(val_edges))
print("Validation edges (negative):", len(val_edges_false))
print("Test edges (positive):", len(test_edges))
print("Test edges (negative):", len(test_edges_false))

Total nodes: 18879
Total edges: 183923
Training edges (positive): 110355
Training edges (negative): 110355
Validation edges (positive): 18392
Validation edges (negative): 18392
Test edges (positive): 55176
Test edges (negative): 55176


In [19]:
from node2vec import Node2Vec

# node2vec settings
# NOTE: When p = q = 1, this is equivalent to DeepWalk
dimensions = 128 
walk_length = 30 # Length of walk per source
num_walks = 10 # Number of walks per source
window = 10 # Window size
workers = 1 # Num. parallel workers
p = 1 # Return hyperparameter
q = 1 # In-out hyperparameter


In [20]:
# Preprocessing, generate walks and train 
node2vec = Node2Vec(g_train, dimensions=dimensions, walk_length=walk_length, num_walks=num_walks, workers=workers, p=p, q=q)
model = node2vec.fit(window=window, min_count=1, batch_words=4)

# Store embeddings mapping
emb_mappings = model.wv




Computing transition probabilities:   0%|          | 0/18879 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 7/18879 [00:00<11:26, 27.49it/s]

Computing transition probabilities:   0%|          | 8/18879 [00:00<30:25, 10.34it/s]

Computing transition probabilities:   0%|          | 30/18879 [00:00<21:45, 14.44it/s]

Computing transition probabilities:   0%|          | 44/18879 [00:00<16:29, 19.03it/s]

Computing transition probabilities:   0%|          | 64/18879 [00:00<12:03, 25.99it/s]

Computing transition probabilities:   0%|          | 75/18879 [00:01<09:22, 33.45it/s]

Computing transition probabilities:   0%|          | 85/18879 [00:01<08:17, 37.79it/s]

Computing transition probabilities:   1%|          | 102/18879 [00:01<06:24, 48.86it/s]

Computing transition probabilities:   1%|          | 115/18879 [00:01<05:37, 55.54it/s]

Computing transition probabilities:   1%|          | 125/18879 [00:01<04:53, 63.79it/s]

Computing transition probabilities:   

Computing transition probabilities:   5%|▌         | 978/18879 [00:12<02:23, 124.97it/s]

Computing transition probabilities:   5%|▌         | 995/18879 [00:13<02:17, 129.76it/s]

Computing transition probabilities:   5%|▌         | 1013/18879 [00:13<02:15, 132.12it/s]

Computing transition probabilities:   5%|▌         | 1031/18879 [00:13<02:17, 130.13it/s]

Computing transition probabilities:   6%|▌         | 1046/18879 [00:13<02:55, 101.51it/s]

Computing transition probabilities:   6%|▌         | 1059/18879 [00:13<03:06, 95.50it/s] 

Computing transition probabilities:   6%|▌         | 1071/18879 [00:13<03:04, 96.78it/s]

Computing transition probabilities:   6%|▌         | 1090/18879 [00:13<02:41, 110.40it/s]

Computing transition probabilities:   6%|▌         | 1103/18879 [00:14<03:59, 74.31it/s] 

Computing transition probabilities:   6%|▌         | 1114/18879 [00:14<03:41, 80.28it/s]

Computing transition probabilities:   6%|▌         | 1125/18879 [00:14<03:30, 84.32it/s]

Comp

Computing transition probabilities:  13%|█▎        | 2471/18879 [00:26<01:41, 161.66it/s]

Computing transition probabilities:  13%|█▎        | 2502/18879 [00:27<01:27, 186.54it/s]

Computing transition probabilities:  13%|█▎        | 2531/18879 [00:27<01:20, 203.89it/s]

Computing transition probabilities:  14%|█▎        | 2558/18879 [00:27<01:16, 214.16it/s]

Computing transition probabilities:  14%|█▍        | 2603/18879 [00:27<01:04, 251.70it/s]

Computing transition probabilities:  14%|█▍        | 2639/18879 [00:27<01:11, 228.28it/s]

Computing transition probabilities:  14%|█▍        | 2666/18879 [00:27<01:17, 208.36it/s]

Computing transition probabilities:  14%|█▍        | 2699/18879 [00:27<01:11, 227.82it/s]

Computing transition probabilities:  14%|█▍        | 2737/18879 [00:27<01:03, 253.22it/s]

Computing transition probabilities:  15%|█▍        | 2765/18879 [00:28<01:21, 197.22it/s]

Computing transition probabilities:  15%|█▍        | 2795/18879 [00:28<01:13, 217.71it/s]


Computing transition probabilities:  67%|██████▋   | 12737/18879 [00:37<00:01, 3407.79it/s]

Computing transition probabilities:  70%|██████▉   | 13125/18879 [00:37<00:01, 3506.43it/s]

Computing transition probabilities:  72%|███████▏  | 13559/18879 [00:37<00:01, 3654.67it/s]

Computing transition probabilities:  74%|███████▍  | 13945/18879 [00:37<00:02, 2157.61it/s]

Computing transition probabilities:  76%|███████▌  | 14379/18879 [00:37<00:01, 2507.32it/s]

Computing transition probabilities:  78%|███████▊  | 14713/18879 [00:39<00:08, 470.73it/s] 

Computing transition probabilities:  79%|███████▉  | 14952/18879 [00:43<00:25, 155.99it/s]

Computing transition probabilities:  80%|████████  | 15121/18879 [00:46<00:38, 98.81it/s] 

Computing transition probabilities:  81%|████████  | 15241/18879 [00:48<00:43, 82.73it/s]

Computing transition probabilities:  81%|████████  | 15327/18879 [00:50<00:52, 68.14it/s]

Computing transition probabilities:  82%|████████▏ | 15389/18879 [00:52<00:5

Computing transition probabilities:  85%|████████▍ | 16001/18879 [01:04<00:49, 58.05it/s]

Computing transition probabilities:  85%|████████▍ | 16008/18879 [01:04<01:04, 44.42it/s]

Computing transition probabilities:  85%|████████▍ | 16014/18879 [01:04<01:07, 42.49it/s]

Computing transition probabilities:  85%|████████▍ | 16019/18879 [01:04<01:17, 36.89it/s]

Computing transition probabilities:  85%|████████▍ | 16025/18879 [01:04<01:16, 37.54it/s]

Computing transition probabilities:  85%|████████▍ | 16033/18879 [01:04<01:06, 43.08it/s]

Computing transition probabilities:  85%|████████▍ | 16039/18879 [01:04<01:00, 46.68it/s]

Computing transition probabilities:  85%|████████▍ | 16046/18879 [01:05<00:56, 50.31it/s]

Computing transition probabilities:  85%|████████▌ | 16052/18879 [01:05<00:54, 51.56it/s]

Computing transition probabilities:  85%|████████▌ | 16058/18879 [01:05<00:59, 47.50it/s]

Computing transition probabilities:  85%|████████▌ | 16064/18879 [01:05<00:57, 49.34it/s]


Computing transition probabilities:  88%|████████▊ | 16545/18879 [01:15<00:44, 52.83it/s]

Computing transition probabilities:  88%|████████▊ | 16552/18879 [01:15<00:43, 53.98it/s]

Computing transition probabilities:  88%|████████▊ | 16558/18879 [01:15<00:45, 51.23it/s]

Computing transition probabilities:  88%|████████▊ | 16566/18879 [01:15<00:41, 55.77it/s]

Computing transition probabilities:  88%|████████▊ | 16572/18879 [01:15<00:41, 55.47it/s]

Computing transition probabilities:  88%|████████▊ | 16579/18879 [01:15<00:39, 58.84it/s]

Computing transition probabilities:  88%|████████▊ | 16586/18879 [01:16<00:39, 58.36it/s]

Computing transition probabilities:  88%|████████▊ | 16592/18879 [01:16<00:40, 55.87it/s]

Computing transition probabilities:  88%|████████▊ | 16598/18879 [01:16<00:42, 53.07it/s]

Computing transition probabilities:  88%|████████▊ | 16606/18879 [01:16<00:39, 57.94it/s]

Computing transition probabilities:  88%|████████▊ | 16613/18879 [01:16<00:37, 60.15it/s]


Computing transition probabilities:  91%|█████████ | 17144/18879 [01:26<00:31, 54.58it/s]

Computing transition probabilities:  91%|█████████ | 17150/18879 [01:26<00:34, 50.64it/s]

Computing transition probabilities:  91%|█████████ | 17156/18879 [01:26<00:35, 48.33it/s]

Computing transition probabilities:  91%|█████████ | 17162/18879 [01:26<00:37, 46.06it/s]

Computing transition probabilities:  91%|█████████ | 17167/18879 [01:26<00:42, 39.86it/s]

Computing transition probabilities:  91%|█████████ | 17172/18879 [01:26<00:47, 36.08it/s]

Computing transition probabilities:  91%|█████████ | 17176/18879 [01:27<00:50, 33.47it/s]

Computing transition probabilities:  91%|█████████ | 17180/18879 [01:27<00:51, 33.00it/s]

Computing transition probabilities:  91%|█████████ | 17184/18879 [01:27<00:49, 34.58it/s]

Computing transition probabilities:  91%|█████████ | 17190/18879 [01:27<00:45, 36.78it/s]

Computing transition probabilities:  91%|█████████ | 17199/18879 [01:27<00:39, 42.68it/s]


Computing transition probabilities:  94%|█████████▎| 17695/18879 [01:37<00:34, 34.01it/s]

Computing transition probabilities:  94%|█████████▍| 17701/18879 [01:38<00:30, 38.32it/s]

Computing transition probabilities:  94%|█████████▍| 17706/18879 [01:38<00:29, 40.22it/s]

Computing transition probabilities:  94%|█████████▍| 17711/18879 [01:38<00:29, 39.15it/s]

Computing transition probabilities:  94%|█████████▍| 17720/18879 [01:38<00:24, 47.14it/s]

Computing transition probabilities:  94%|█████████▍| 17726/18879 [01:38<00:27, 41.21it/s]

Computing transition probabilities:  94%|█████████▍| 17731/18879 [01:38<00:26, 42.98it/s]

Computing transition probabilities:  94%|█████████▍| 17736/18879 [01:38<00:26, 43.64it/s]

Computing transition probabilities:  94%|█████████▍| 17741/18879 [01:38<00:26, 42.85it/s]

Computing transition probabilities:  94%|█████████▍| 17751/18879 [01:38<00:22, 50.79it/s]

Computing transition probabilities:  94%|█████████▍| 17757/18879 [01:39<00:27, 41.13it/s]


Computing transition probabilities:  96%|█████████▌| 18161/18879 [01:48<00:18, 37.79it/s]

Computing transition probabilities:  96%|█████████▌| 18165/18879 [01:49<00:20, 34.43it/s]

Computing transition probabilities:  96%|█████████▌| 18171/18879 [01:49<00:18, 38.17it/s]

Computing transition probabilities:  96%|█████████▋| 18176/18879 [01:49<00:18, 38.05it/s]

Computing transition probabilities:  96%|█████████▋| 18182/18879 [01:49<00:17, 40.18it/s]

Computing transition probabilities:  96%|█████████▋| 18187/18879 [01:49<00:17, 38.80it/s]

Computing transition probabilities:  96%|█████████▋| 18194/18879 [01:49<00:15, 44.08it/s]

Computing transition probabilities:  96%|█████████▋| 18199/18879 [01:49<00:20, 33.78it/s]

Computing transition probabilities:  96%|█████████▋| 18203/18879 [01:50<00:21, 31.06it/s]

Computing transition probabilities:  96%|█████████▋| 18207/18879 [01:50<00:23, 29.05it/s]

Computing transition probabilities:  96%|█████████▋| 18211/18879 [01:50<00:21, 31.39it/s]


Computing transition probabilities:  99%|█████████▉| 18657/18879 [02:00<00:05, 42.63it/s]

Computing transition probabilities:  99%|█████████▉| 18662/18879 [02:00<00:07, 31.00it/s]

Computing transition probabilities:  99%|█████████▉| 18666/18879 [02:00<00:06, 32.52it/s]

Computing transition probabilities:  99%|█████████▉| 18672/18879 [02:00<00:05, 35.09it/s]

Computing transition probabilities:  99%|█████████▉| 18678/18879 [02:00<00:05, 37.39it/s]

Computing transition probabilities:  99%|█████████▉| 18683/18879 [02:01<00:05, 37.70it/s]

Computing transition probabilities:  99%|█████████▉| 18688/18879 [02:01<00:05, 31.96it/s]

Computing transition probabilities:  99%|█████████▉| 18693/18879 [02:01<00:05, 35.42it/s]

Computing transition probabilities:  99%|█████████▉| 18699/18879 [02:01<00:04, 39.42it/s]

Computing transition probabilities:  99%|█████████▉| 18707/18879 [02:01<00:03, 45.68it/s]

Computing transition probabilities:  99%|█████████▉| 18713/18879 [02:01<00:03, 47.60it/s]


In [21]:
# Create node embeddings matrix (rows = nodes, columns = embedding features)
emb_list = []
for node_index in range(0, adj_sparse.shape[0]):
    node_str = str(node_index)
    node_emb = emb_mappings[node_str]
    emb_list.append(node_emb)
emb_matrix = np.vstack(emb_list)

In [22]:
# Generate bootstrapped edge embeddings (as is done in node2vec paper)
    # Edge embedding for (v1, v2) = hadamard product of node embeddings for v1, v2
def get_edge_embeddings(edge_list):
    embs = []
    for edge in edge_list:
        node1 = edge[0]
        node2 = edge[1]
        emb1 = emb_matrix[node1]
        emb2 = emb_matrix[node2]
        edge_emb = np.multiply(emb1, emb2)
        embs.append(edge_emb)
    embs = np.array(embs)
    return embs

In [23]:
# Train-set edge embeddings
pos_train_edge_embs = get_edge_embeddings(train_edges)
neg_train_edge_embs = get_edge_embeddings(train_edges_false)
train_edge_embs = np.concatenate([pos_train_edge_embs, neg_train_edge_embs])

# Create train-set edge labels: 1 = real edge, 0 = false edge
train_edge_labels = np.concatenate([np.ones(len(train_edges)), np.zeros(len(train_edges_false))])

# Val-set edge embeddings, labels
pos_val_edge_embs = get_edge_embeddings(val_edges)
neg_val_edge_embs = get_edge_embeddings(val_edges_false)
val_edge_embs = np.concatenate([pos_val_edge_embs, neg_val_edge_embs])
val_edge_labels = np.concatenate([np.ones(len(val_edges)), np.zeros(len(val_edges_false))])

# Test-set edge embeddings, labels
pos_test_edge_embs = get_edge_embeddings(test_edges)
neg_test_edge_embs = get_edge_embeddings(test_edges_false)
test_edge_embs = np.concatenate([pos_test_edge_embs, neg_test_edge_embs])

# Create val-set edge labels: 1 = real edge, 0 = false edge
test_edge_labels = np.concatenate([np.ones(len(test_edges)), np.zeros(len(test_edges_false))])

In [24]:
# Train logistic regression classifier on train-set edge embeddings
from sklearn.linear_model import LogisticRegression
edge_classifier = LogisticRegression(random_state=0)
edge_classifier.fit(train_edge_embs, train_edge_labels)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=0, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [25]:
# Predicted edge scores: probability of being of class "1" (real edge)
val_preds = edge_classifier.predict_proba(val_edge_embs)[:, 1]
val_roc = roc_auc_score(val_edge_labels, val_preds)
val_ap = average_precision_score(val_edge_labels, val_preds)

In [26]:
# Predicted edge scores: probability of being of class "1" (real edge)
test_preds = edge_classifier.predict_proba(test_edge_embs)[:, 1]
test_roc = roc_auc_score(test_edge_labels, test_preds)
test_ap = average_precision_score(test_edge_labels, test_preds)

In [27]:
print('node2vec Validation ROC score: ', str(val_roc))
print('node2vec Validation AP score: ', str(val_ap))
print('node2vec Test ROC score: ', str(test_roc))
print('node2vec Test AP score: ', str(test_ap))

node2vec Validation ROC score:  0.8937518441126794
node2vec Validation AP score:  0.92391845918454
node2vec Test ROC score:  0.8912392013344347
node2vec Test AP score:  0.923167800312918
